# Week 8 – E-Commerce Order Analytics System

## Part 1: Data Generation

This notebook generates realistic e-commerce datasets with intentional data quality issues for downstream cleaning and SQL analysis.

## Imports & Configuration

In [1]:
"""
Week 8 Mini Project - E-Commerce Order Analytics System
Part 1: Data Generation

Generates 4 related CSVs (customers, products, orders, order_items) with
realistic-looking data AND deliberately baked-in messiness, because the
whole point of this project is to clean/validate data that looks like it
came from a real, slightly broken production system.

Everything that's "wrong" in the data is wrong on purpose - controlled by
the CONFIG block below so the amount of mess can be dialed up or down
without touching the generation logic itself.
"""

import csv
import random
from datetime import datetime, timedelta

from faker import Faker

fake = Faker()

# ---------------------------------------------------------------------------
# CONFIG - tweak these instead of hunting through the functions below
# ---------------------------------------------------------------------------
SEED = 42                      # set to None for a fresh random dataset every run
NUM_CUSTOMERS = 600
NUM_PRODUCTS = 600
NUM_ORDERS = 2500

REFERENCE_DATE = datetime(2026, 7, 19)   # "today" for the fake business
ORDER_HISTORY_MONTHS = 26                # how far back orders go (need >12 for YoY queries)

# intentional issue rates (fractions of the relevant table)
PCT_NULL_CUSTOMER_ID = 0.05
PCT_ORDER_BAD_DATE_FORMAT = 0.05
PCT_INVALID_EMAIL = 0.02
PCT_MESSY_PRODUCT_NAME = 0.12
PCT_NEGATIVE_QTY = 0.03

# small fixed counts of "seed" rows for edge-case testing later (Part 5)
NUM_BAD_ORDER_ID_REFS = 8         # order_items pointing at an order_id that doesn't exist
NUM_DISCOUNT_OVER_100 = 6         # discount_percent > 100 (invalid, on purpose)
NUM_ZERO_QUANTITY = 5             # quantity == 0
NUM_FUTURE_DATED_ORDERS = 5       # order_date after REFERENCE_DATE

from pathlib import Path

OUTPUT_DIR = Path("../raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SEED is not None:
    random.seed(SEED)
    Faker.seed(SEED)

# ---------------------------------------------------------------------------
# Reference data - not hardcoded product lists, just the building blocks used
# to *generate* varied combinations dynamically
# ---------------------------------------------------------------------------
CATEGORY_SUBCATEGORIES = {
    "Electronics": ["Mobile Phones", "Laptops", "Headphones", "Smart Watches", "Cameras", "Televisions"],
    "Clothing": ["Men's Wear", "Women's Wear", "Kids Wear", "Footwear", "Accessories"],
    "Home": ["Kitchen", "Furniture", "Decor", "Bedding", "Storage"],
    "Books": ["Fiction", "Non-Fiction", "Academic", "Comics", "Children"],
    "Sports": ["Fitness Gear", "Outdoor", "Team Sports", "Cycling"],
    "Beauty": ["Skincare", "Haircare", "Makeup", "Fragrance"],
}

# typical cost-price bands per category, used to keep prices realistic instead
# of pure random noise (an Electronics item shouldn't cost the same as a Book)
CATEGORY_COST_RANGE = {
    "Electronics": (800, 45000),
    "Clothing": (150, 3500),
    "Home": (200, 8000),
    "Books": (80, 900),
    "Sports": (250, 6000),
    "Beauty": (100, 2500),
}

BRAND_WORDS = ["Nova", "Zenith", "Urban", "Prime", "Aero", "Crest", "Vibe",
               "Nimbus", "Orbit", "Pulse", "Terra", "Lumen", "Vertex", "Drift"]
DESCRIPTOR_WORDS = ["Pro", "Max", "Lite", "Plus", "Classic", "Elite", "Essential",
                    "Ultra", "Basic", "Signature", "Everyday", "Premium"]

# explicit singular noun per subcategory, used to build product names.
# earlier version tried to auto-derive this with subcategory.split()[0].rstrip("s"),
# which silently mangled anything ending in a double-s ("Fitness" -> "Fitne")
# or a possessive ("Women's" -> "Women'"). A fixed lookup avoids that entirely.
SUBCATEGORY_NOUN = {
    "Mobile Phones": "Phone", "Laptops": "Laptop", "Headphones": "Headphone",
    "Smart Watches": "Watch", "Cameras": "Camera", "Televisions": "Television",
    "Men's Wear": "Shirt", "Women's Wear": "Dress", "Kids Wear": "Outfit",
    "Footwear": "Shoe", "Accessories": "Accessory",
    "Kitchen": "Cookware", "Furniture": "Chair", "Decor": "Decor Piece",
    "Bedding": "Bedsheet", "Storage": "Organizer",
    "Fiction": "Novel", "Non-Fiction": "Guide", "Academic": "Textbook",
    "Comics": "Comic", "Children": "Storybook",
    "Fitness Gear": "Tracker", "Outdoor": "Tent", "Team Sports": "Jersey", "Cycling": "Bicycle",
    "Skincare": "Cream", "Haircare": "Shampoo", "Makeup": "Lipstick", "Fragrance": "Perfume",
}

REGION_CODES = ["NORTH", "SOUTH", "EAST", "WEST", "CENTRAL"]
ORDER_STATUSES = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
STATUS_WEIGHTS = [0.10, 0.15, 0.55, 0.10, 0.10]
CUSTOMER_TYPES = ["REGULAR", "PREMIUM", "VIP"]
CUSTOMER_TYPE_WEIGHTS = [0.65, 0.25, 0.10]




## Helper Function: random_datetime_between

In [2]:
def random_datetime_between(start: datetime, end: datetime) -> datetime:
    """Uniform random datetime between two datetimes."""
    delta_seconds = int((end - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, max(delta_seconds, 1)))




## Helper Function: make_messy_product_name

In [3]:
def make_messy_product_name(name: str) -> str:
    """Randomly scramble casing/spacing to mimic sloppy manual data entry."""
    variant = random.choice(["extra_spaces", "upper", "lower", "both"])
    if variant in ("extra_spaces", "both"):
        name = "  " + "  ".join(name.split()) + "  "
    if variant == "upper":
        name = name.upper()
    elif variant == "lower":
        name = name.lower()
    return name




## Helper Function: make_invalid_email

In [4]:
def make_invalid_email(valid_email: str) -> str:
    """Break an otherwise valid email in one of a few realistic ways."""
    local, _, domain = valid_email.partition("@")
    mode = random.choice(["no_at", "no_domain", "double_at"])
    if mode == "no_at":
        return f"{local}{domain}"
    if mode == "no_domain":
        return f"{local}@"
    return f"{local}@@{domain}"


# ---------------------------------------------------------------------------
# Generators
# ---------------------------------------------------------------------------


## Helper Function: generate_customers

In [5]:
def generate_customers(n):
    rows = []
    for cust_id in range(1, n + 1):
        name = fake.name()
        email = fake.unique.email()
        if random.random() < PCT_INVALID_EMAIL:
            email = make_invalid_email(email)

        reg_date = fake.date_between(
            start_date=REFERENCE_DATE - timedelta(days=730),
            end_date=REFERENCE_DATE - timedelta(days=1),
        )
        cust_type = random.choices(CUSTOMER_TYPES, weights=CUSTOMER_TYPE_WEIGHTS, k=1)[0]

        rows.append({
            "customer_id": cust_id,
            "customer_name": name,
            "email": email,
            "registration_date": reg_date.strftime("%Y-%m-%d"),
            "customer_type": cust_type,
        })
    return rows




## Helper Function: generate_products

In [6]:
def generate_products(n):
    rows = []
    for prod_id in range(1, n + 1):
        category = random.choice(list(CATEGORY_SUBCATEGORIES.keys()))
        subcategory = random.choice(CATEGORY_SUBCATEGORIES[category])

        brand = random.choice(BRAND_WORDS)
        descriptor = random.choice(DESCRIPTOR_WORDS)
        noun = SUBCATEGORY_NOUN[subcategory]
        name = f"{brand} {noun} {descriptor}"

        if random.random() < PCT_MESSY_PRODUCT_NAME:
            name = make_messy_product_name(name)

        low, high = CATEGORY_COST_RANGE[category]
        cost_price = round(random.uniform(low, high), 2)

        rows.append({
            "product_id": prod_id,
            "product_name": name,
            "category": category,
            "subcategory": subcategory,
            "cost_price": cost_price,
        })
    return rows




## Helper Function: generate_orders

In [7]:
def generate_orders(n, customer_ids):
    rows = []
    start = REFERENCE_DATE - timedelta(days=ORDER_HISTORY_MONTHS * 30)

    future_order_slots = set(random.sample(range(1, n + 1), NUM_FUTURE_DATED_ORDERS))

    for order_id in range(1, n + 1):
        customer_id = random.choice(customer_ids)
        if random.random() < PCT_NULL_CUSTOMER_ID:
            customer_id = ""  # written as blank -> NULL when loaded

        if order_id in future_order_slots:
            order_dt = REFERENCE_DATE + timedelta(days=random.randint(1, 60))
        else:
            order_dt = random_datetime_between(start, REFERENCE_DATE)

        status = random.choices(ORDER_STATUSES, weights=STATUS_WEIGHTS, k=1)[0]
        region = random.choice(REGION_CODES)

        if random.random() < PCT_ORDER_BAD_DATE_FORMAT:
            date_str = order_dt.strftime("%d-%m-%Y")  # wrong format, no time component
        else:
            date_str = order_dt.strftime("%Y-%m-%d %H:%M:%S")

        rows.append({
            "order_id": order_id,
            "customer_id": customer_id,
            "order_date": date_str,
            "status": status,
            "region_code": region,
        })
    return rows




## Helper Function: generate_order_items

In [8]:
def generate_order_items(orders, products):
    rows = []
    item_id = 1
    valid_order_ids = [o["order_id"] for o in orders]
    product_lookup = {p["product_id"]: p for p in products}
    product_ids = list(product_lookup.keys())

    for order in orders:
        num_items = random.choices([1, 2, 3, 4], weights=[0.45, 0.30, 0.17, 0.08], k=1)[0]
        for _ in range(num_items):
            product = product_lookup[random.choice(product_ids)]
            quantity = random.randint(1, 5)
            if random.random() < PCT_NEGATIVE_QTY:
                quantity = -random.randint(1, 3)  # a return

            markup = random.uniform(1.2, 1.8)
            unit_price = round(product["cost_price"] * markup, 2)
            discount_percent = random.choices(
                [0, random.randint(1, 15), random.randint(16, 40)],
                weights=[0.3, 0.4, 0.3], k=1
            )[0]

            rows.append({
                "item_id": item_id,
                "order_id": order["order_id"],
                "product_id": product["product_id"],
                "quantity": quantity,
                "unit_price": unit_price,
                "discount_percent": discount_percent,
            })
            item_id += 1

    # --- seed rows for edge-case testing (Part 5) ---
    max_valid_order_id = max(valid_order_ids)

    # 1. order_items referencing an order_id that doesn't exist
    for _ in range(NUM_BAD_ORDER_ID_REFS):
        product = product_lookup[random.choice(product_ids)]
        rows.append({
            "item_id": item_id,
            "order_id": max_valid_order_id + random.randint(1000, 9000),
            "product_id": product["product_id"],
            "quantity": random.randint(1, 3),
            "unit_price": round(product["cost_price"] * 1.4, 2),
            "discount_percent": random.randint(0, 20),
        })
        item_id += 1

    # 2. discount_percent > 100 (invalid on purpose)
    for _ in range(NUM_DISCOUNT_OVER_100):
        product = product_lookup[random.choice(product_ids)]
        rows.append({
            "item_id": item_id,
            "order_id": random.choice(valid_order_ids),
            "product_id": product["product_id"],
            "quantity": random.randint(1, 3),
            "unit_price": round(product["cost_price"] * 1.4, 2),
            "discount_percent": random.randint(101, 150),
        })
        item_id += 1

    # 3. quantity == 0
    for _ in range(NUM_ZERO_QUANTITY):
        product = product_lookup[random.choice(product_ids)]
        rows.append({
            "item_id": item_id,
            "order_id": random.choice(valid_order_ids),
            "product_id": product["product_id"],
            "quantity": 0,
            "unit_price": round(product["cost_price"] * 1.4, 2),
            "discount_percent": random.randint(0, 20),
        })
        item_id += 1

    return rows




## Helper Function: write_csv

In [9]:
def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)




## Helper Function: main

In [10]:
def main():
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Generating customers...")
    customers = generate_customers(NUM_CUSTOMERS)
    write_csv(f"{OUTPUT_DIR}/customers.csv", customers,
              ["customer_id", "customer_name", "email", "registration_date", "customer_type"])

    print("Generating products...")
    products = generate_products(NUM_PRODUCTS)
    write_csv(f"{OUTPUT_DIR}/products.csv", products,
              ["product_id", "product_name", "category", "subcategory", "cost_price"])

    print("Generating orders...")
    customer_ids = [c["customer_id"] for c in customers]
    orders = generate_orders(NUM_ORDERS, customer_ids)
    write_csv(f"{OUTPUT_DIR}/orders.csv", orders,
              ["order_id", "customer_id", "order_date", "status", "region_code"])

    print("Generating order_items...")
    order_items = generate_order_items(orders, products)
    write_csv(f"{OUTPUT_DIR}/order_items.csv", order_items,
              ["item_id", "order_id", "product_id", "quantity", "unit_price", "discount_percent"])

    print(f"\nDone. Rows generated:")
    print(f"  customers.csv   : {len(customers)}")
    print(f"  products.csv    : {len(products)}")
    print(f"  orders.csv      : {len(orders)}")
    print(f"  order_items.csv : {len(order_items)}")




## Execution

In [11]:
if __name__ == "__main__":
    main()


Generating customers...
Generating products...
Generating orders...
Generating order_items...

Done. Rows generated:
  customers.csv   : 600
  products.csv    : 600
  orders.csv      : 2500
  order_items.csv : 4710


## Validation & Preview

Review the generated CSVs and verify intentional inconsistencies.

In [12]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../raw")

for file in [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv"
]:
    path = RAW_DIR / file

    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{'='*60}")
        print(f"{file}")
        print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
        display(df.head())
    else:
        print(f"{file} not found in {RAW_DIR}")


customers.csv
Rows: 600 | Columns: 5


,customer_id,customer_name,email,registration_date,customer_type
0,1,Allison Hill,donaldgarcia@example.net,2026-06-11,REGULAR
1,2,Leslie Johnson,robinsonwilliam@example.org,2025-09-27,REGULAR
2,3,Matthew Gardner,shaneramirez@example.org,2026-02-28,PREMIUM
3,4,Melissa Peterson,jasongallagher@example.org,2024-09-29,REGULAR
4,5,Ian Cooper,lindsay78@example.org,2026-07-05,REGULAR



products.csv
Rows: 600 | Columns: 5


,product_id,product_name,category,subcategory,cost_price
0,1,Vertex Tent Elite,Sports,Outdoor,5759.78
1,2,Prime Shampoo Max,Beauty,Haircare,573.45
2,3,Vertex Tent Everyday,Sports,Outdoor,2917.73
3,4,Terra Bicycle Signature,Sports,Cycling,5922.56
4,5,Urban Decor Piece Ultra,Home,Decor,5124.80



orders.csv
Rows: 2,500 | Columns: 5


,order_id,customer_id,order_date,status,region_code
0,1,241.0,2025-07-16 07:01:46,SHIPPED,SOUTH
1,2,365.0,2024-11-11 17:39:54,SHIPPED,NORTH
2,3,505.0,2024-08-03 15:41:00,CANCELLED,EAST
3,4,505.0,2025-07-30 18:47:23,DELIVERED,CENTRAL
4,5,228.0,2024-12-15 05:21:20,PLACED,WEST



order_items.csv
Rows: 4,710 | Columns: 6


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,1,1,115,5,2467.77,0
1,2,1,198,-3,3061.70,5
2,3,2,116,3,556.73,34
3,4,2,115,1,2202.34,3
4,5,3,285,5,1389.35,3


In [13]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../raw")

customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")

print("=" * 60)
print("DATA VALIDATION SUMMARY")
print("=" * 60)

# Dataset sizes
print(f"\nCustomers     : {customers.shape}")
print(f"Products      : {products.shape}")
print(f"Orders        : {orders.shape}")
print(f"Order Items   : {order_items.shape}")

# Missing customer IDs
missing_customer = orders["customer_id"].isna().sum()
print(f"\nMissing Customer IDs: {missing_customer} ({missing_customer/len(orders)*100:.2f}%)")

# Negative quantities
negative_qty = (order_items["quantity"] < 0).sum()
print(f"Negative Quantities: {negative_qty} ({negative_qty/len(order_items)*100:.2f}%)")

# Invalid emails
invalid_email = ~customers["email"].str.contains(r"^[^@]+@[^@]+\.[^@]+$", regex=True, na=False)
print(f"Invalid Emails: {invalid_email.sum()} ({invalid_email.mean()*100:.2f}%)")

# Mixed date formats
wrong_dates = orders["order_date"].astype(str).str.match(r"\d{2}-\d{2}-\d{4}")
print(f"Wrong Date Format: {wrong_dates.sum()}")

# Product names with extra spaces
extra_spaces = products["product_name"].astype(str).str.contains(r"^\s|\s$", regex=True)
print(f"Extra Spaces in Product Names: {extra_spaces.sum()}")

# Referential integrity
invalid_orders = ~order_items["order_id"].isin(orders["order_id"])
print(f"Broken Order References: {invalid_orders.sum()}")

print("\n Validation Complete")

DATA VALIDATION SUMMARY

Customers     : (600, 5)
Products      : (600, 5)
Orders        : (2500, 5)
Order Items   : (4710, 6)

Missing Customer IDs: 119 (4.76%)
Negative Quantities: 150 (3.18%)
Invalid Emails: 14 (2.33%)
Wrong Date Format: 113
Extra Spaces in Product Names: 41
Broken Order References: 8

 Validation Complete
